# 09a — Fetch Spanish mobile-phone OD data in R

**Kernel: R**

The `spanishoddata` package is used to download Spanish district-zone definitions and aggregated mobile-phone OD records.

This notebook performs data acquisition only. Downstream descriptive analysis is performed in Python.

The mobile-phone rows are **stratified aggregate records**, not individual trajectories. `n_trips` is the weighted trip count.

In [ ]:
# Project-specific R library used on CSC/Roihu.
r_lib <- "/projappl/project_2010417/project_rpackages_452"

dir.create(
  r_lib,
  recursive = TRUE,
  showWarnings = FALSE
)

.libPaths(c(r_lib, .libPaths()))
.libPaths()

In [ ]:
required_packages <- c(
  "spanishoddata",
  "dplyr",
  "sf",
  "arrow"
)

missing_packages <- required_packages[
  !sapply(required_packages, requireNamespace, quietly = TRUE)
]

if (length(missing_packages) > 0) {
  stop(
    "Missing R packages: ",
    paste(missing_packages, collapse = ", "),
    ". Install them before continuing."
  )
}

library(spanishoddata)
library(dplyr)
library(sf)
library(arrow)

## 1. Configure the project data folders

In [ ]:
project_root <- "/scratch/project_2010417/paper3/heat-conditioned-accessibility"

mobile_raw <- file.path(project_root, "data/mobile/raw")
mobile_interim <- file.path(project_root, "data/mobile/interim")

dir.create(mobile_raw, recursive = TRUE, showWarnings = FALSE)
dir.create(mobile_interim, recursive = TRUE, showWarnings = FALSE)

spod_set_data_dir(mobile_raw)
spod_get_data_dir()

## 2. Download district zones and isolate Sevilla municipality

In [ ]:
districts <- spod_get_zones(
  zones = "districts",
  ver = 2
)

# Sevilla municipality INE code.
sevilla_zones <- districts |>
  filter(municipalities == "41091")

sevilla_ids <- sevilla_zones$id

sevilla_zones |>
  select(
    id,
    name,
    population,
    municipalities,
    luas_mitma
  )

In [ ]:
st_write(
  sevilla_zones,
  file.path(
    mobile_interim,
    "sevilla_districts_v2.gpkg"
  ),
  delete_dsn = TRUE
)

## 3. Download one diagnostic day

In [ ]:
test_date <- "2024-07-15"

od_test <- spod_get(
  type = "od",
  zones = "districts",
  dates = test_date
)

sevilla_od_test <- od_test |>
  filter(
    id_origin %in% sevilla_ids |
    id_destination %in% sevilla_ids
  ) |>
  collect()

arrow::write_parquet(
  sevilla_od_test,
  file.path(
    mobile_interim,
    "sevilla_od_2024-07-15.parquet"
  )
)

dim(sevilla_od_test)

## 4. Download the hot/reference-day sample

In [ ]:
study_dates <- c(
  "2024-06-27",
  "2024-06-28",
  "2024-07-02",
  "2024-07-03",
  "2024-07-18",
  "2024-07-19",
  "2024-07-23",
  "2024-07-24",
  "2024-07-25"
)

od_multi <- spod_get(
  type = "od",
  zones = "districts",
  dates = study_dates
)

sevilla_od_multi <- od_multi |>
  filter(
    id_origin %in% sevilla_ids |
    id_destination %in% sevilla_ids
  ) |>
  collect()

arrow::write_parquet(
  sevilla_od_multi,
  file.path(
    mobile_interim,
    "sevilla_od_heat_reference_sample.parquet"
  )
)

dim(sevilla_od_multi)